In [0]:
%sql
use catalog ext_cat

In [0]:
%sql
drop table if exists orders;

In [0]:
%sql
create table orders(
    order_id int,
    order_date string,
    customer_id int,
    order_status string
)using delta

Without inferSchema, order_id/customer_id come in as string, conflicting with the table's int columns.
With inferSchema='true', Spark correctly infers order_id/customer_id as int, but also infers order_date as date (from date-formatted values), conflicting with the table's string column.

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        order_id,
        cast(order_date as string) as order_date,
        customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
    -- 'inferSchema'='true'
)
/* copy_options not required as schema is fixed */

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5427491746295877>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "copy into ext_cat.default.orders\nfrom (\n    select\n        order_id,\n        cast(order_date as string) as order_date,\n        customer_id,\n        order_status\n    from '/Volumes/ext_cat/default/extvol/emp'\n)\nfileformat = CSV\nformat_options(\n    'header' = 'true'\n    -- 'inferSchema'='true'\n)\n/* copy_options not required as schema is fixed */\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_s

Below are 2 options - either inferschema and them cast date col as string, or do not inferschema and cast numeric vals to int as per table definition|

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        order_id,
        cast(order_date as string) as order_date,
        customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true',
    'inferSchema'='true'
)
/* copy_options not required as schema is fixed */

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
/* copy_options not required as schema is fixed */

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


Add a new file to source location and rerun

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
/* copy_options not required as schema is fixed */

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


In [0]:
%sql
select count(*) from orders

count(1)
18


Upload a new file with a new col an try

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
/* copy_options not required as schema is fixed */


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


Data was loaded, but the additional column  was ignored

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status
10101,25-08-2025,2101,Delivered
10102,26-08-2025,2105,Shipped
10103,27-08-2025,2102,Processing
10104,28-08-2025,2108,Cancelled
10105,30-08-2025,2101,Delivered
10106,01-09-2025,2110,Shipped
10107,02-09-2025,2104,Processing
10108,03-09-2025,2107,Pending
10109,05-09-2025,1203,Pending
1001,2026-08-25,201,Delivered


Now add a file with same cols but different data type

In [0]:
spark.conf.get('spark.sql.ansi.enabled')

'false'

a null was loaded due to  invalid int value

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
/* copy_options not required as schema is fixed */


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


Case 1: ANSI mode ON (this is the default for Databricks SQL warehouses on accounts created after Oct 19, 2022)
The rules and behaviors regarding CAST are stricter in ANSI mode, and for many conversions Databricks SQL uses clear SQL data type casting rules. Concretely, when spark.sql.ansi.enabled is set to true, explicit casting by CAST syntax throws a runtime exception for illegal cast patterns defined in the standard, such as casts from a string to an integer. So if a row has customer_id = "abc", the command throws something like CAST_INVALID_INPUT: The value 'abc' of type STRING cannot be cast to INT because it is malformed, and the whole COPY INTO command fails — no rows get loaded from that file (and since COPY INTO tracks file-level idempotency, once you fix it you'd need to either fix the file or make sure it hasn't been marked as already processed). 
Databricks
Databricks

Case 2: ANSI mode OFF (classic/legacy behavior, still the default for spark.sql.ansi.enabled on many notebook/cluster runtimes prior to DBR 17.0)
Here CAST is lenient: SELECT cast('a' AS INT) returns null instead of erroring. So a bad customer_id value silently becomes NULL in the loaded row — the command succeeds, but you get silent data loss/corruption: that row loads with a null customer_id, and you won't get any error to tell you it happened.

In [0]:
%sql
drop table if exists ext_cat.default.orders;
create table ext_cat.default.orders(
    order_id int,
    order_date string,
    customer_id int,
    order_status string
)using delta

Now load again with mergeSchema copy options

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
copy_options(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


In [0]:
%sql
select * from ext_cat.default.orders

order_id,order_date,customer_id,order_status
1001,2026-08-25,201,Delivered
1002,2026-08-26,205,Shipped
1003,2026-08-27,202,Processing
1004,2026-08-28,208,Cancelled
1005,2026-08-30,201,Delivered
1006,2026-09-01,210,Shipped
1007,2026-09-02,204,Processing
1008,2026-09-03,207,Pending
1009,2026-09-05,203,Pending


In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
copy_options(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
9,9,0


Since during the copy into table from -select statement didnt mention the new column , it was not loaded even with mergeSchema=true

In [0]:
%sql
select * from ext_cat.default.orders

order_id,order_date,customer_id,order_status
1001,2026-08-25,201,Delivered
1002,2026-08-26,205,Shipped
1003,2026-08-27,202,Processing
1004,2026-08-28,208,Cancelled
1005,2026-08-30,201,Delivered
1006,2026-09-01,210,Shipped
1007,2026-09-02,204,Processing
1008,2026-09-03,207,Pending
1009,2026-09-05,203,Pending
10101,25-08-2025,null,Delivered


for loading metadata

In [0]:
%sql
drop table if exists ext_cat.default.orders;
create table ext_cat.default.orders(
    order_id int,
    order_date string,
    customer_id int,
    order_status string,
    _file_name string,
    _load_date timestamp
)using delta

In [0]:
%sql
copy into ext_cat.default.orders
from (
    select
        cast(order_id as int ) as order_id,
        order_date,
        cast (customer_id as int ) as customer_id,
        order_status,
        _metadata.file_name as _file_name,
        current_timestamp() as _load_date 
    from '/Volumes/ext_cat/default/extvol/emp'
)
fileformat = CSV
format_options(
    'header' = 'true'
)
copy_options(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
18,18,0


In [0]:
%sql
select * from ext_cat.default.orders

order_id,order_date,customer_id,order_status,_file_name,_load_date
1001,2026-08-25,201,Delivered,orders1.csv,2026-09-05T13:34:00.222579Z
1002,2026-08-26,205,Shipped,orders1.csv,2026-09-05T13:34:00.222579Z
1003,2026-08-27,202,Processing,orders1.csv,2026-09-05T13:34:00.222579Z
1004,2026-08-28,208,Cancelled,orders1.csv,2026-09-05T13:34:00.222579Z
1005,2026-08-30,201,Delivered,orders1.csv,2026-09-05T13:34:00.222579Z
1006,2026-09-01,210,Shipped,orders1.csv,2026-09-05T13:34:00.222579Z
1007,2026-09-02,204,Processing,orders1.csv,2026-09-05T13:34:00.222579Z
1008,2026-09-03,207,Pending,orders1.csv,2026-09-05T13:34:00.222579Z
1009,2026-09-05,203,Pending,orders1.csv,2026-09-05T13:34:00.222579Z
10101,25-08-2025,null,Delivered,orders5.csv,2026-09-05T13:34:00.222579Z
